# 12 — Model Invocation Patterns

Three ways to call a LangChain LLM:

| Method | When | Returns |
|---|---|---|
| `.invoke(x)` | One-shot answer | Full response |
| `.stream(x)` | Real-time UX | Iterator of chunks |
| `.batch([x])` | Many prompts | List of responses |

**Config source:** `configs/default.yaml` → `llm`

In [ ]:
from rag_pipeline.utils import load_notebook_config
from rag_pipeline.generation import build_llms

cfg, REPO = load_notebook_config()
llms = build_llms(dict(cfg.llm))
model_name = next(iter(llms))
llm = llms[model_name]
print("Using model:", model_name)

**`.invoke()` — single prompt, blocking**

In [ ]:
q = "Explain in two sentences what a graph neural network is."
print(f"Q: {q}\nA: {llm.invoke(q, max_new_tokens=80)}")

**`.stream()` — chunk-by-chunk**

In [ ]:
q = "List three key challenges in federated learning."
print(f"Q: {q}\nA: ", end="")
for chunk in llm.stream(q, max_new_tokens=120):
    print(str(chunk), end="", flush=True)
print()

**`.batch()` — parallel processing**

In [ ]:
questions = [
    "What is meta-learning?",
    "Explain the attention mechanism.",
    "What are GANs used for?",
]
responses = llm.batch(questions, max_new_tokens=80)
for i, (q, a) in enumerate(zip(questions, responses), 1):
    print(f"{i}. Q: {q}\n   A: {str(a).strip()[:120]}...\n")

**Chat-message format**

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are a concise research assistant."),
    HumanMessage(content="What is contrastive learning? Answer in one sentence."),
]
print("AI:", llm.invoke(messages, max_new_tokens=60))

**Batch with concurrency control**

In [ ]:
batch_items = [
    "Define self-supervised learning.",
    "What is reinforcement learning?",
    "Explain transfer learning.",
    "What is multi-task learning?",
]
responses = llm.batch(batch_items, max_new_tokens=60, config={"max_concurrency": 2})
for i, r in enumerate(responses, 1):
    print(f"{i}. {str(r).strip()[:100]}...")